# Práctica 05: análisis del dataset clínico simulado de Puebla

Este notebook valida y analiza 5,000 registros completamente ficticios. El indicador de riesgo es académico y no constituye un diagnóstico médico.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
BASE = Path.cwd()
if BASE.name == 'notebooks':
    BASE = BASE.parent
elif not (BASE / 'data').exists() and (BASE / 'Practica05').exists():
    BASE = BASE / 'Practica05'
RUTA_CSV = BASE / 'data' / 'pacientes_puebla_5000.csv'
RUTA_OUTPUTS = BASE / 'outputs'
RUTA_OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f'Directorio de la práctica: {BASE.resolve()}')

## 1. Carga e inspección inicial

In [ ]:
df = pd.read_csv(RUTA_CSV, dtype={'id_paciente': 'string'})
df.head()

In [ ]:
print(f'Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas')
assert df.shape == (5000, 22), 'Las dimensiones no son las esperadas.'
df.columns.tolist()

In [ ]:
df.dtypes.to_frame('tipo_de_dato')

## 2. Calidad de datos

In [ ]:
nulos = df.isna().sum()
duplicados_fila = df.duplicated().sum()
duplicados_id = df['id_paciente'].duplicated().sum()
print('Valores nulos por columna:')
display(nulos.to_frame('nulos'))
print(f'Filas duplicadas: {duplicados_fila}')
print(f'Identificadores duplicados: {duplicados_id}')
assert nulos.sum() == 0 and duplicados_fila == 0 and duplicados_id == 0

In [ ]:
rangos = {
    'edad': (18, 90), 'peso_kg': (35, 170), 'estatura_m': (1.45, 1.95),
    'imc': (17, 42.1), 'presion_sistolica': (90, 190),
    'presion_diastolica': (55, 120), 'glucosa_mg_dl': (65, 240),
    'colesterol_mg_dl': (110, 330), 'frecuencia_cardiaca_lpm': (50, 120)
}
validacion_rangos = pd.DataFrame({
    columna: {'mínimo_observado': df[columna].min(), 'máximo_observado': df[columna].max(),
              'fuera_de_rango': (~df[columna].between(minimo, maximo)).sum()}
    for columna, (minimo, maximo) in rangos.items()
}).T
display(validacion_rangos)
assert validacion_rangos['fuera_de_rango'].sum() == 0
assert (df['presion_sistolica'] - df['presion_diastolica']).ge(20).all()

In [ ]:
catalogo_geo = pd.DataFrame([
    ('Puebla', 'Heroica Puebla de Zaragoza', 19.0414, -98.2063),
    ('Tehuacán', 'Tehuacán', 18.4615, -97.3928),
    ('San Martín Texmelucan', 'San Martín Texmelucan de Labastida', 19.2843, -98.4389),
    ('Atlixco', 'Atlixco', 18.9089, -98.4361),
    ('San Pedro Cholula', 'Cholula de Rivadavia', 19.0641, -98.3035),
    ('Huauchinango', 'Huauchinango', 20.1767, -98.0528),
    ('Teziutlán', 'Teziutlán', 19.8175, -97.3599),
    ('Amozoc', 'Amozoc de Mota', 19.0450, -98.0442),
    ('Izúcar de Matamoros', 'Izúcar de Matamoros', 18.6016, -98.4654),
    ('Xicotepec', 'Xicotepec de Juárez', 20.2750, -97.9611),
    ('Zacatlán', 'Zacatlán', 19.9348, -97.9613),
    ('Cuautlancingo', 'San Juan Cuautlancingo', 19.0895, -98.2732),
    ('Tecamachalco', 'Tecamachalco', 18.8814, -97.7336),
    ('Acatlán', 'Acatlán de Osorio', 18.2025, -98.0486),
    ('Chignahuapan', 'Chignahuapan', 19.8380, -98.0317),
], columns=['municipio', 'localidad', 'lat_centro', 'lon_centro'])
geo = df.merge(catalogo_geo, on=['municipio', 'localidad'], how='left', validate='many_to_one')
pares_invalidos = geo['lat_centro'].isna().sum()
coordenadas_alejadas = ((geo['latitud'] - geo['lat_centro']).abs() > 0.0181) | ((geo['longitud'] - geo['lon_centro']).abs() > 0.0181)
print(f'Pares municipio-localidad no válidos: {pares_invalidos}')
print(f'Coordenadas fuera del entorno esperado: {coordenadas_alejadas.sum()}')
assert pares_invalidos == 0 and coordenadas_alejadas.sum() == 0

## 3. Limpieza básica y estadísticas

In [ ]:
# No se detectaron problemas. Aun así, se aplican operaciones básicas seguras.
df_limpio = df.drop_duplicates().copy()
columnas_texto = df_limpio.select_dtypes(include=['object', 'string']).columns
for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].str.strip()
df_limpio['imc_calculado'] = (df_limpio['peso_kg'] / df_limpio['estatura_m'].pow(2)).round(1)
diferencia_imc = (df_limpio['imc'] - df_limpio['imc_calculado']).abs().max()
print(f'Filas después de limpieza: {len(df_limpio)}')
print(f'Máxima diferencia de IMC por redondeo: {diferencia_imc:.1f}')
assert len(df_limpio) == 5000 and diferencia_imc <= 0.1

In [ ]:
df_limpio.describe(include='all').T

## 4. Distribución del riesgo y visualizaciones

In [ ]:
orden_riesgo = ['bajo', 'medio', 'alto']
distribucion = df_limpio['riesgo_cardiovascular'].value_counts().reindex(orden_riesgo)
porcentaje = (distribucion / len(df_limpio) * 100).round(2)
pd.DataFrame({'pacientes': distribucion, 'porcentaje': porcentaje})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df_limpio, x='edad', bins=18, color='#2878B5', ax=axes[0])
axes[0].set_title('Distribución de edad')
sns.histplot(data=df_limpio, x='imc', bins=20, color='#F28E2B', ax=axes[1])
axes[1].set_title('Distribución del IMC')
fig.tight_layout()
fig.savefig(RUTA_OUTPUTS / 'histogramas_edad_imc.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df_limpio, x='riesgo_cardiovascular', order=orden_riesgo, hue='riesgo_cardiovascular', palette='RdYlGn_r', legend=False, ax=ax)
ax.set(title='Distribución del riesgo cardiovascular', xlabel='Nivel de riesgo', ylabel='Pacientes')
fig.tight_layout()
fig.savefig(RUTA_OUTPUTS / 'barras_riesgo_cardiovascular.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
conteo_municipios = df_limpio['municipio'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
conteo_municipios.plot.barh(color='#59A14F', ax=ax)
ax.set(title='Pacientes simulados por municipio', xlabel='Pacientes', ylabel='Municipio')
fig.tight_layout()
fig.savefig(RUTA_OUTPUTS / 'barras_municipios.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
muestra = df_limpio.sample(1200, random_state=230389)
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=muestra, x='edad', y='presion_sistolica', hue='riesgo_cardiovascular', hue_order=orden_riesgo, palette='RdYlGn_r', alpha=0.65, ax=ax)
ax.set(title='Edad y presión sistólica por nivel de riesgo', xlabel='Edad (años)', ylabel='Presión sistólica (mmHg)')
fig.tight_layout()
fig.savefig(RUTA_OUTPUTS / 'dispersion_edad_presion.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## 5. Confirmación final

In [ ]:
graficas = sorted(ruta.name for ruta in RUTA_OUTPUTS.glob('*.png'))
assert len(df_limpio) == 5000
assert df_limpio.isna().sum().sum() == 0
assert df_limpio.duplicated().sum() == 0
assert set(df_limpio['riesgo_cardiovascular']) == set(orden_riesgo)
assert len(graficas) >= 4
print('Validación completada correctamente.')
print('Gráficas guardadas:', graficas)